# DRS × glass packaging recycling: country–year panel

**Question:** Do financial incentives such as a deposit return system (DRS) increase glass packaging recycling rates?

**What this notebook builds:** one tidy table with one row per country per year (2000–2023), combining
1. **Outcome**: Eurostat glass packaging recycling rate (`cei_wm020_glass_page_jsonstat.json`), cleaned of impossible values (see below)
2. **DRS timing for glass**: the year glass entered each country's DRS, typed in below (Reloop Global Deposit Book 2024, Reloop deck, and operator sources where checked)
3. **DRS intensity**: deposit amount for glass from the Reloop 2020 and 2022 editions (`deposit_return_schemes_2020_2022.csv`)

**Put the two input files in the `data/` folder** next to this notebook (or change `DATA_DIR` below).

**What changed compared with the plastic notebook**
- The start year is **material-specific** (`glass_drs_start`): Finland is treated for glass from the year glass joined, the Netherlands' system has no glass and only covers cans from April 2023, and Sweden, Norway and Slovakia have no glass.
- `drs_status` separates countries **with no DRS at all** from countries that **have a DRS that does not cover glass**. Both have `has_drs = 0`, but they are different comparison groups.
- **The deposit is the deposit on glass**, not the highest deposit of the country. For countries whose deposit varies by material (Denmark, Finland, Norway, Sweden) it is only known from the 2022 edition (2021), because the 2020 edition does not split by material.
- No kg-per-capita volumes and no return rate: the volumes file and the CSV's return rate are plastic only.

**Data cleaning.** Glass packaging (bottles, but also jars and other non-drink glass) has values that cannot be real recycling rates. The published value is kept in `glass_recycling_rate_raw`; `glass_recycling_rate` is the analysis version with these removed:
- **Above 100%**: recycling cannot exceed what was generated; this usually means recycling is counted in a different year or on a different basis than generation.
- **Identical value for 4+ consecutive years** (e.g. exactly 100% for 13–18 years): a reporting convention, not a measurement.
- **Below 1%**: the material is effectively not measured (for example Iceland's glass).
- A `glass_recycling_rate_capped` column (raw values capped at 100) is provided for sensitivity checks, and every exclusion is listed in section 4 with its reason in `dq_flag`.

**Caveat specific to glass:** Glass is the weakest material for a before/after design. Several countries already had deposits on refillable glass bottles long before their DRS (Finland since 1950), so the start year marks the DRS for one-way glass, not the first financial incentive on glass. Four DRS countries (Sweden, Norway, Netherlands, Slovakia) do not cover glass at all, so they act as within-country comparisons.

In [ ]:
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

MATERIAL = "glass"
WASTE_CODE = "W150107"          # Eurostat code for this packaging material
SCOPE_REGEX = r"\bglass\b"           # how this material is named in the Reloop scope text

DATA_DIR = Path(os.environ.get("DRS_DATA_DIR", "data"))        # folder with the input files
OUT_DIR = Path(os.environ.get("DRS_OUT_DIR", str(DATA_DIR)))   # where the outputs are written
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTCOME_FILE = DATA_DIR / f"cei_wm020_{MATERIAL}_page_jsonstat.json"
DEPOSIT_CSV = DATA_DIR / "deposit_return_schemes_2020_2022.csv"

AGGREGATES = ["EU27_2020", "EA21", "EA20", "EA19"]   # Eurostat totals, not countries
RATE_UNIT = "RT_TGT2025"

# --- data-cleaning switches (see the cleaning section in the introduction) ---------------
EXCLUDE_OVER_100 = False      # rates above 100% are set to missing in the analysis column
EXCLUDE_FLAT_RUNS = True     # identical value for RUN_MIN_LEN+ consecutive years -> missing
EXCLUDE_NEAR_ZERO = False     # rates below NEAR_ZERO_BELOW -> missing
RUN_MIN_LEN = 4
NEAR_ZERO_BELOW = 1.0
MANUAL_EXCLUDE = []          # extra exclusions as (geo, first_year, last_year, "reason")

# Approximate EUR per 1 unit of local currency, used only to make deposits comparable across countries.
# Rough period averages, NOT sourced. Replace with Eurostat annual averages (table ert_bil_eur_a)
# if the euro value matters for your analysis. National-currency values stay in the snapshot table.
FX_EUR_PER_LOCAL = {
    "EUR": 1.0,
    "HRK": 1 / 7.5,
    "DKK": 1 / 7.46,
    "NOK": 1 / 10.2,
    "SEK": 1 / 10.4,
    "ISK": 1 / 147.0,
}

## 1. Read the Eurostat file

In [2]:
def parse_jsonstat(path):
    """Turn a Eurostat JSON-stat 2.0 file into a long DataFrame:
    one column per dimension + 'value' + 'flag' (Eurostat status code)."""
    with open(path, encoding="utf-8") as f:
        d = json.load(f)

    ids, size = d["id"], d["size"]
    codes, labels = [], {}
    for dim in ids:
        cat = d["dimension"][dim]["category"]
        idx = cat["index"]
        order = sorted(idx, key=idx.get) if isinstance(idx, dict) else list(idx)
        codes.append(order)
        labels[dim] = cat.get("label", {})

    n = int(np.prod(size))
    values = np.full(n, np.nan)
    v = d["value"]
    if isinstance(v, dict):                      # sparse form: {flat_index: value}
        for k, x in v.items():
            values[int(k)] = x
    else:                                        # dense form: list with nulls
        values[:] = [np.nan if x is None else x for x in v]

    flags = np.full(n, "", dtype=object)
    s = d.get("status", {})
    if isinstance(s, dict):
        for k, x in s.items():
            flags[int(k)] = x
    elif isinstance(s, list):
        for k, x in enumerate(s):
            flags[k] = x or ""

    index = pd.MultiIndex.from_product(codes, names=ids)
    df = pd.DataFrame({"value": values, "flag": flags}, index=index).reset_index()
    return df, labels


raw, labels = parse_jsonstat(OUTCOME_FILE)
assert raw["waste"].unique().tolist() == [WASTE_CODE], raw["waste"].unique()   # right material file?
RAW = f"{MATERIAL}_recycling_rate_raw"
outcome = (
    raw[(raw["waste"] == WASTE_CODE) & (raw["unit"] == RATE_UNIT)]
    .rename(columns={"value": RAW, "flag": "rate_flag"})
    [["geo", "time", RAW, "rate_flag"]]
    .assign(year=lambda x: x["time"].astype(int))
    .drop(columns="time")
)
country_names = labels["geo"]
print("outcome rows:", len(outcome), "| years:", outcome["year"].min(), "-", outcome["year"].max())
outcome.head()

outcome rows: 984 | years: 2000 - 2023


,geo,glass_recycling_rate_raw,rate_flag,year
0,EU27_2020,NaN,,2000
1,EU27_2020,NaN,,2001
2,EU27_2020,NaN,,2002
3,EU27_2020,NaN,,2003
4,EU27_2020,NaN,,2004


## 2. Clean the outcome

Three kinds of observation are removed from the analysis column (the raw value is always kept):

| Rule | Why |
|---|---|
| Rate > 100% | A recycling rate cannot exceed 100% of what was generated |
| Same value for 4+ consecutive years | A reporting convention (a constant 100% or 0%), not a measurement |
| Rate < 1% | The material is effectively not measured in that country-year |

Switches and thresholds are in the config cell.

In [3]:
RATE = f"{MATERIAL}_recycling_rate"
rate = outcome[~outcome["geo"].isin(AGGREGATES)].sort_values(["geo", "year"]).reset_index(drop=True)

rate["dq_over_100"] = rate[RAW] > 100
rate["dq_near_zero"] = rate[RAW] < NEAR_ZERO_BELOW

# runs of identical values in consecutive years
rate["dq_flat_run"] = False
for geo, g in rate.dropna(subset=[RAW]).groupby("geo"):
    vals, yrs, ids = g[RAW].round(1).to_numpy(), g["year"].to_numpy(), g.index.to_numpy()
    start = 0
    for i in range(1, len(vals) + 1):
        if i == len(vals) or vals[i] != vals[start] or yrs[i] != yrs[i - 1] + 1:
            if i - start >= RUN_MIN_LEN:
                rate.loc[ids[start:i], "dq_flat_run"] = True
            start = i

rate["dq_manual"] = False
for geo, y0, y1, reason in MANUAL_EXCLUDE:
    rate.loc[(rate["geo"] == geo) & rate["year"].between(y0, y1), "dq_manual"] = True

def dq_label(r):
    out = []
    if r["dq_over_100"]: out.append("over_100")
    if r["dq_flat_run"]: out.append("flat_run")
    if r["dq_near_zero"]: out.append("near_zero")
    if r["dq_manual"]: out.append("manual")
    return "+".join(out)

rate["dq_flag"] = rate.apply(dq_label, axis=1)
rate["rate_excluded"] = (
    (EXCLUDE_OVER_100 & rate["dq_over_100"]) | (EXCLUDE_FLAT_RUNS & rate["dq_flat_run"])
    | (EXCLUDE_NEAR_ZERO & rate["dq_near_zero"]) | rate["dq_manual"]
)
rate[RATE] = rate[RAW].where(~rate["rate_excluded"])
rate[f"{RATE}_capped"] = rate[RAW].clip(upper=100)

print(f"observations with a published rate: {rate[RAW].notna().sum()} | excluded: {int(rate['rate_excluded'].sum())}"
      f" | left for analysis: {rate[RATE].notna().sum()}")
print("\nWhat was excluded, by country:")
display(
    rate[rate["rate_excluded"]]
    .groupby("geo")
    .apply(lambda g: pd.Series({
        "n": len(g),
        "years": f"{g['year'].min()}-{g['year'].max()}" if len(g) > 1 else str(g['year'].iloc[0]),
        "reasons": ", ".join(sorted(set("+".join(g["dq_flag"]).split("+")))),
        "raw_values": ", ".join(f"{v:.0f}" for v in g[RAW].head(6)) + (" ..." if len(g) > 6 else ""),
    }), include_groups=False)
)

observations with a published rate: 630 | excluded: 40 | left for analysis: 590

What was excluded, by country:


,n,years,reasons,raw_values
geo,,,,
BE,13,2007-2019,flat_run,"100, 100, 100, 100, 100, 100 ..."
DK,6,2004-2010,over_100,"103, 115, 128, 121, 174, 174"
EE,1,2019,over_100,105
IS,13,2010-2022,"flat_run, near_zero","0, 0, 0, 0, 0, 0 ..."
MT,1,2008,over_100,142
NO,2,2006-2008,over_100,"106, 101"
SE,1,2004,over_100,104
SI,3,2016-2023,over_100,"100, 104, 112"


## 3. DRS timing (country level, material-specific)

Start years come from the **Reloop Global Deposit Book 2024** (`drs_start_used` is the DRS as a whole), with the older Reloop deck year alongside for comparison. **`metal_start` and `glass_start` give the year each material entered the DRS; an empty value means the material is not in that country's DRS.** Where operator sources were checked this is noted (Finland: Palpa; Netherlands: Verpact).

Countries whose DRS starts after 2023 (HU, IE, AT, PL, PT) are listed so they are correctly treated as *not yet treated*; their material coverage is not verified and does not affect results.

In [4]:
drs = pd.DataFrame(
    [
        # geo, DRS start (GDB 2024), start (deck), metal start, glass start, note
        ("SE", 1984, 1984, 1984, None, "Cans first (1984); glass is not in the national DRS (Reloop CSV and deck). PET start year not checked"),
        ("IS", 1989, 1989, 1989, 1989, "Deposit increases 2002, 2010, Jan 2021, Jan 2023"),
        ("FI", 1996, 1996, 1996, 2011, "Cans 1996, PET 2008, one-way glass 2011 per Palpa (GDB 2024 says 2012). Refillable glass bottles have had deposits since 1950"),
        ("NO", 1999, 1999, 1999, None, "Plastic and metal only; deposit increased in 2018"),
        ("DK", 2002, 2002, 2002, 2002, "Scope expanded 2005, 2008, 2020 (drink types)"),
        ("DE", 2003, 2003, 2003, 2003, "Scope expanded 2022 and 2024 (drink types)"),
        ("EE", 2005, 2005, 2005, 2005, "Deposit increases June 2010, July 2011, Feb 2015"),
        ("NL", 2007, 2005, 2023, None, "Cans from 1 April 2023 (Verpact). Glass: only refillable beer bottles, not part of the DRS. Deck says 2005, GDB 2024 says 2007 for the system"),
        ("HR", 2006, 2006, 2006, 2006, "0.50 HRK until 2022; EUR 0.07 from 2023"),
        ("LT", 2016, 2016, 2016, 2016, "In-window switcher"),
        ("MT", 2022, 2020, 2022, 2022, "Deck says 2020, GDB 2024 says 2022"),
        ("SK", 2022, 2022, 2022, None, "Plastic and metal only (Reloop 2022 edition)"),
        ("LV", 2022, 2022, 2022, 2022, "Started 1 Feb 2022"),
        ("RO", 2023, 2022, 2023, 2023, "Deck says 2022, GDB 2024 says 2023. Scope is not in the CSV: metal and glass coverage ASSUMED, verify"),
        ("PT", 2026, 2022, 2026, 2026, "Deck says 2022; GDB 2024 lists it as upcoming (2026). Scope not verified; outside the window"),
        ("HU", 2024, None, 2024, 2024, "Outside the data window; scope not verified"),
        ("IE", 2024, None, 2024, 2024, "Outside the data window; scope not verified"),
        ("AT", 2025, None, 2025, 2025, "Outside the data window; scope not verified"),
        ("PL", 2025, None, 2025, 2025, "Planned for 2025, possible delay; outside the data window; scope not verified"),
    ],
    columns=["geo", "drs_start_used", "drs_start_deck", "metal_start", "glass_start", "drs_note"],
)

# keep only this material's start year, named like the outcome
drs[f"{MATERIAL}_drs_start"] = drs[f"{MATERIAL}_start"]
drs = drs.drop(columns=["metal_start", "glass_start"])

def check(row):
    if pd.isna(row["drs_start_deck"]):
        return "not in deck"
    return "agrees with deck" if row["drs_start_used"] == row["drs_start_deck"] else "DIFFERS from deck"

drs["drs_start_check"] = drs.apply(check, axis=1)
START = f"{MATERIAL}_drs_start"
drs

,geo,drs_start_used,drs_start_deck,drs_note,glass_drs_start,drs_start_check
0,SE,1984,1984.0,Cans first (1984); glass is not in the nationa...,NaN,agrees with deck
1,IS,1989,1989.0,"Deposit increases 2002, 2010, Jan 2021, Jan 2023",1989.0,agrees with deck
2,FI,1996,1996.0,"Cans 1996, PET 2008, one-way glass 2011 per Pa...",2011.0,agrees with deck
3,NO,1999,1999.0,Plastic and metal only; deposit increased in 2018,NaN,agrees with deck
4,DK,2002,2002.0,"Scope expanded 2005, 2008, 2020 (drink types)",2002.0,agrees with deck
5,DE,2003,2003.0,Scope expanded 2022 and 2024 (drink types),2003.0,agrees with deck
6,EE,2005,2005.0,"Deposit increases June 2010, July 2011, Feb 2015",2005.0,agrees with deck
7,NL,2007,2005.0,Cans from 1 April 2023 (Verpact). Glass: only ...,NaN,DIFFERS from deck
8,HR,2006,2006.0,0.50 HRK until 2022; EUR 0.07 from 2023,2006.0,agrees with deck
9,LT,2016,2016.0,In-window switcher,2016.0,agrees with deck


## 4. DRS intensity: deposit on this material (Reloop 2020 and 2022 editions)

The CSV covers deposit systems worldwide; only the European ones can be matched to Eurostat. How the deposit for **this material** is read:
- **Flat systems** (one deposit for everything): that amount, in both editions.
- **Variable systems, 2022 edition** (Denmark, Finland, Norway, Sweden): the amounts listed for this material.
- **Variable systems, 2020 edition**: the text gives one range for all materials, so the deposit for this material is **left empty** rather than guessed.
- A snapshot is only used if its scope text mentions this material.
- Snapshots are carried forward to the next snapshot and flagged (`deposit_is_snapshot_year`). Deposits are known only from about 2018.

In [5]:
NAME2GEO = {
    "Croatia": "HR", "Denmark": "DK", "Estonia": "EE", "Finland": "FI", "Germany": "DE",
    "Iceland": "IS", "Latvia": "LV", "Lithuania": "LT", "Malta": "MT", "Netherlands": "NL",
    "The Netherlands": "NL", "Norway": "NO", "Slovakia": "SK", "Sweden": "SE",
}
# Manual corrections to the scope text, where the CSV wording misleads
SCOPE_OVERRIDE = {}

def parse_year(v):
    m = re.match(r"\s*(\d{4})", str(v))
    return int(m.group(1)) if m else np.nan

def find_amounts(segment):
    vals = []
    for m in re.finditer(r"(?<![\d.])(\d+(?:\.\d+)?)\s*(HRK|DKK|ISK|NOK|SEK)\b", segment):
        vals.append((float(m.group(1)), m.group(2)))
    for m in re.finditer(r"€\s*(\d+(?:\.\d+)?)", segment):
        vals.append((float(m.group(1)), "EUR"))
    return vals

def parse_refund(text):
    out = dict(currency=None, deposit_structure=None, mat_min_local=np.nan, mat_max_local=np.nan)
    if pd.isna(text):
        return out
    t = str(text).strip()
    main = re.sub(r"\([^)]*\)", " ", t)          # drop bracketed conversions and "(VAT incl.)"
    vals = find_amounts(main)
    if not vals:
        return out
    cur = vals[0][1]
    nums = [v for v, c in vals if c == cur]
    out["currency"] = cur
    if t.startswith("Variable"):
        out["deposit_structure"] = "variable"
        segs = [s for s in main.split(";") if re.search(SCOPE_REGEX, s, re.I)]
        mat = [v for s in segs for v, c in find_amounts(s) if c == cur]
        if mat:                                    # 2022 edition lists amounts by material and size
            out["mat_min_local"], out["mat_max_local"] = min(mat), max(mat)
    elif t.startswith("Fixed") or min(nums) == max(nums):
        out["deposit_structure"] = "flat"          # same deposit for every material
        out["mat_min_local"], out["mat_max_local"] = min(nums), max(nums)
    else:
        out["deposit_structure"] = "variable (range)"   # 2020 edition: one range, no split by material
    return out

csv = pd.read_csv(DEPOSIT_CSV, encoding="utf-8-sig")
csv.columns = ["region", "year_raw", "scope_text", "refund_text", "return_rate_text", "mechanism", "edition"]
csv["geo"] = csv["region"].map(NAME2GEO)
eu = csv[csv["geo"].notna()].copy()
print(f"kept {len(eu)} of {len(csv)} rows (non-European systems dropped); countries: {sorted(eu['geo'].unique())}")

parsed = pd.DataFrame([parse_refund(t) for t in eu["refund_text"]], index=eu.index)
snap = pd.concat([eu, parsed], axis=1)
snap["snapshot_year"] = snap["year_raw"].apply(parse_year)

# rows without a year: date them to the DRS start year and flag
start_map = drs.set_index("geo")[START].fillna(drs.set_index("geo")["drs_start_used"])
snap["snapshot_year_imputed"] = snap["snapshot_year"].isna()
snap.loc[snap["snapshot_year_imputed"], "snapshot_year"] = snap.loc[snap["snapshot_year_imputed"], "geo"].map(start_map)
snap["snapshot_year"] = snap["snapshot_year"].astype(int)

snap["covers_material"] = snap["scope_text"].str.contains(SCOPE_REGEX, case=False, regex=True)
for geo, flag in SCOPE_OVERRIDE.items():
    snap.loc[snap["geo"] == geo, "covers_material"] = flag
snap["edition"] = "Reloop " + snap["edition"].astype(str)

# Values from other sources that extend the CSV (add more rows here if you find them)
MANUAL = [
    # geo, year, currency, deposit (local), note
    ("IS", 2023, "ISK", 20.0, "Reloop GDB 2024: flat ISK 20 after the January 2023 increase (all materials)"),
]
manual = pd.DataFrame(MANUAL, columns=["geo", "snapshot_year", "currency", "deposit_local", "refund_text"])
manual["mat_min_local"] = manual["deposit_local"]
manual["mat_max_local"] = manual["deposit_local"]
manual["deposit_structure"] = "flat"
manual["covers_material"] = True
manual["snapshot_year_imputed"] = False
manual["edition"] = "manual (other source)"
snap = pd.concat([snap, manual.drop(columns="deposit_local")], ignore_index=True)

fx = snap["currency"].map(FX_EUR_PER_LOCAL)
snap["deposit_min_eur"] = snap["mat_min_local"] * fx
snap["deposit_max_eur"] = snap["mat_max_local"] * fx

snap_cols = ["geo", "edition", "snapshot_year", "snapshot_year_imputed", "covers_material", "deposit_structure",
             "currency", "mat_min_local", "mat_max_local", "deposit_min_eur", "deposit_max_eur",
             "scope_text", "refund_text"]
snap = snap[snap_cols].sort_values(["geo", "snapshot_year"]).reset_index(drop=True)
snap

kept 23 of 96 rows (non-European systems dropped); countries: ['DE', 'DK', 'EE', 'FI', 'HR', 'IS', 'LT', 'LV', 'MT', 'NL', 'NO', 'SE', 'SK']


,geo,edition,snapshot_year,snapshot_year_imputed,covers_material,deposit_structure,currency,mat_min_local,mat_max_local,deposit_min_eur,deposit_max_eur,scope_text,refund_text
0,DE,Reloop 2020,2018,False,True,flat,EUR,0.25,0.25,0.250000,0.250000,"Plastic, metal, glass (containers 100ml-3L)",€0.25 (USD$0.28)
1,DE,Reloop 2022,2021,False,True,flat,EUR,0.25,0.25,0.250000,0.250000,"Plastic, metal, glass",Fixed: €0.25
2,DK,Reloop 2020,2019,False,True,variable (range),DKK,NaN,NaN,NaN,NaN,"Plastic, metal, glass","1 DKK (€0.13, USD$0.15) to 3 DKK (€0.40, USD$0..."
3,DK,Reloop 2022,2021,False,True,variable,DKK,1.00,3.00,0.134048,0.402145,"Plastic, metal, glass","Variable: Metal, glass <1L: 1 DKK; Plastic <1L..."
4,EE,Reloop 2020,2019,False,True,flat,EUR,0.10,0.10,0.100000,0.100000,"Plastic, metal, glass (containers 100ml-3L)",€0.10 (USD$0.11)
5,EE,Reloop 2022,2020,False,True,flat,EUR,0.10,0.10,0.100000,0.100000,"Plastic, metal, glass",Fixed: €0.10
6,FI,Reloop 2020,2019,False,True,variable (range),EUR,NaN,NaN,NaN,NaN,"Plastic, metal, glass",€0.10 (USD$0.11) to €0.40 (USD$0.45)
7,FI,Reloop 2022,2021,False,True,variable,EUR,0.10,0.10,0.100000,0.100000,"Plastic, metal, glass",Variable: Plastic ≤350ml: €0.10; Plastic 351ml...
8,HR,Reloop 2020,2019,False,True,flat,HRK,0.50,0.50,0.066667,0.066667,"Plastic, metal, glass (containers >200ml)","0.50 HRK (€0.06, USD$0.07)"
9,HR,Reloop 2022,2020,False,True,flat,HRK,0.50,0.50,0.066667,0.066667,"Plastic, metal, glass",Fixed: 0.50 HRK


In [6]:
# Does the CSV scope text agree with the start years typed in section 3?
cons = (snap[snap["edition"] != "manual (other source)"].groupby("geo")["covers_material"].any().rename("csv_says_covered")
        .to_frame().join(drs.set_index("geo")[START].notna().rename("start_year_given")))
print("Countries where the CSV scope and the typed start year disagree (empty = consistent):")
display(cons[cons["csv_says_covered"] != cons["start_year_given"]])

Countries where the CSV scope and the typed start year disagree (empty = consistent):


,csv_says_covered,start_year_given
geo,,


## 5. Assemble the country–year panel

In [7]:
geos = [g for g in country_names if g not in AGGREGATES]
years = sorted(rate["year"].unique())

panel = pd.MultiIndex.from_product([geos, years], names=["geo", "year"]).to_frame(index=False)
panel["country"] = panel["geo"].map(country_names)
panel["material"] = MATERIAL
keep = ["geo", "year", RAW, RATE, f"{RATE}_capped", "rate_flag", "dq_flag", "rate_excluded"]
panel = panel.merge(rate[keep], on=["geo", "year"], how="left")

# Drop geographies with no published rate at all (e.g. CH, UK, Western Balkans, TR in this extract)
has_data = panel.groupby("geo")[RAW].apply(lambda s: s.notna().any())
dropped = has_data[~has_data].index.tolist()
panel = panel[panel["geo"].isin(has_data[has_data].index)].copy()
print("dropped (no published rate):", dropped)

panel = panel.merge(drs, on="geo", how="left")

# --- DRS timing variables (this material) ----------------------------------------------
panel["has_drs"] = (panel["year"] >= panel[START]).fillna(False).astype(int)
panel["drs_full_year"] = (panel["year"] > panel[START]).fillna(False).astype(int)
in_window = panel[START].notna() & (panel[START] <= years[-1])
panel["years_since_drs"] = np.where(in_window, panel["year"] - panel[START], np.nan)

def drs_group(start):
    if pd.isna(start) or start > years[-1]:
        return "3_no_drs_in_window"
    if start <= years[0]:
        return "1_drs_before_window"
    return "2_drs_introduced_in_window"

panel["drs_group"] = panel[START].apply(drs_group)

# a country can have a DRS that does not cover this material
panel["has_any_drs"] = (panel["year"] >= panel["drs_start_used"]).fillna(False).astype(int)
panel["drs_status"] = np.select(
    [panel["has_drs"] == 1, panel["has_any_drs"] == 1],
    [f"{MATERIAL}_in_drs", "other_materials_only"],
    default="no_drs",
)

# --- DRS intensity: deposit on this material, carried forward from each snapshot --------
snap_asof = (snap[snap["covers_material"]].dropna(subset=["deposit_max_eur"])
                 .drop_duplicates(["geo", "snapshot_year"], keep="last")
                 .sort_values("snapshot_year")
                 [["geo", "snapshot_year", "deposit_max_eur", "deposit_min_eur", "deposit_structure"]])
panel = panel.sort_values("year")
panel = pd.merge_asof(panel, snap_asof, left_on="year", right_on="snapshot_year", by="geo", direction="backward")
panel = panel.rename(columns={"snapshot_year": "deposit_source_year"})

no_drs = panel["has_drs"] == 0
panel.loc[no_drs, ["deposit_max_eur", "deposit_min_eur"]] = 0.0     # material not in a DRS = no deposit
panel.loc[no_drs, ["deposit_structure", "deposit_source_year"]] = np.nan
panel["deposit_is_snapshot_year"] = panel["deposit_source_year"] == panel["year"]

# --- measurement flags -------------------------------------------------------------------
panel["series_break_flag"] = panel["rate_flag"].isin(["b", "bd", "be"])
panel["definition_differs_flag"] = panel["rate_flag"].isin(["d", "bd"])
panel["post_2020_method_period"] = (panel["year"] >= 2020).astype(int)

panel = panel.sort_values(["geo", "year"]).reset_index(drop=True)
panel.head(10)

dropped (no published rate): ['AL', 'CH', 'ME', 'MK', 'RS', 'TR', 'UK']


,geo,year,country,material,glass_recycling_rate_raw,glass_recycling_rate,glass_recycling_rate_capped,rate_flag,dq_flag,rate_excluded,...,has_any_drs,drs_status,deposit_source_year,deposit_max_eur,deposit_min_eur,deposit_structure,deposit_is_snapshot_year,series_break_flag,definition_differs_flag,post_2020_method_period
0,AT,2000,Austria,glass,97.4,97.4,97.4,,,False,...,0,no_drs,NaN,0.0,0.0,NaN,False,False,False,0
1,AT,2001,Austria,glass,82.3,82.3,82.3,,,False,...,0,no_drs,NaN,0.0,0.0,NaN,False,False,False,0
2,AT,2002,Austria,glass,86.4,86.4,86.4,,,False,...,0,no_drs,NaN,0.0,0.0,NaN,False,False,False,0
3,AT,2003,Austria,glass,82.9,82.9,82.9,,,False,...,0,no_drs,NaN,0.0,0.0,NaN,False,False,False,0
4,AT,2004,Austria,glass,86.4,86.4,86.4,,,False,...,0,no_drs,NaN,0.0,0.0,NaN,False,False,False,0
5,AT,2005,Austria,glass,79.3,79.3,79.3,,,False,...,0,no_drs,NaN,0.0,0.0,NaN,False,False,False,0
6,AT,2006,Austria,glass,84.5,84.5,84.5,,,False,...,0,no_drs,NaN,0.0,0.0,NaN,False,False,False,0
7,AT,2007,Austria,glass,85.9,85.9,85.9,,,False,...,0,no_drs,NaN,0.0,0.0,NaN,False,False,False,0
8,AT,2008,Austria,glass,84.3,84.3,84.3,,,False,...,0,no_drs,NaN,0.0,0.0,NaN,False,False,False,0
9,AT,2009,Austria,glass,84.8,84.8,84.8,,,False,...,0,no_drs,NaN,0.0,0.0,NaN,False,False,False,0


## 6. Sanity checks

In [8]:
assert not panel.duplicated(["geo", "year"]).any()
print("rows:", len(panel), "| countries:", panel["geo"].nunique(), "| years:", years[0], "-", years[-1])

# 1. After cleaning nothing impossible should remain
print("analysis rates above 100:", int((panel[RATE] > 100).sum()), "| below 1:", int((panel[RATE] < NEAR_ZERO_BELOW).sum()))

# 2. Missing outcome after cleaning
miss = panel[panel[RATE].isna()].groupby("geo")["year"].agg(["count", "min", "max"])
print("country-years with no usable rate:", int(panel[RATE].isna().sum()), "of", len(panel))
display(miss.sort_values("count", ascending=False))

# 3. How many pre/post years does each in-window DRS country actually have?
sw = panel[panel["drs_group"] == "2_drs_introduced_in_window"].copy()
sw["pre"] = (sw["has_drs"] == 0) & sw[RATE].notna()
sw["post"] = (sw["has_drs"] == 1) & sw[RATE].notna()
sw["lost"] = sw["rate_excluded"]
ident = sw.groupby("geo").agg(start=(START, "first"), pre_years=("pre", "sum"), post_years=("post", "sum"),
                              excluded_years=("lost", "sum"), check=("drs_start_check", "first"))
print(f"DRS covering {MATERIAL} introduced inside 2000-2023 (pre/post years with a usable rate):")
display(ident)

# 4. Comparison groups
print("Country-years by DRS status:")
display(panel.groupby("drs_status")["geo"].agg(["count", "nunique"]))

# 5. Deposit coverage among country-years with a DRS in force
d = panel[panel["has_drs"] == 1]
print(f"{MATERIAL} DRS country-years: {len(d)} | with a deposit value: {d['deposit_max_eur'].notna().sum()}"
      f" | of which carried forward from an earlier snapshot: "
      f"{(d['deposit_max_eur'].notna() & ~d['deposit_is_snapshot_year']).sum()}")
display(d.dropna(subset=["deposit_max_eur"]).groupby("geo")["year"].agg(["min", "max"]))

# 6. Group sizes
display(panel.groupby("drs_group")["geo"].nunique().rename("n_countries"))

rows: 720 | countries: 30 | years: 2000 - 2023
analysis rates above 100: 0 | below 1: 0
country-years with no usable rate: 130 of 720


,count,min,max
geo,,,
IS,23,2000,2022
BE,13,2007,2019
HR,13,2000,2023
NO,8,2000,2008
SI,7,2000,2023
PL,7,2000,2023
BG,7,2000,2023
LI,6,2000,2005
DK,6,2004,2010


DRS covering glass introduced inside 2000-2023 (pre/post years with a usable rate):


,start,pre_years,post_years,excluded_years,check
geo,,,,,
DE,2003.0,3,21,0,agrees with deck
DK,2002.0,2,16,6,agrees with deck
EE,2005.0,1,18,1,agrees with deck
FI,2011.0,11,13,0,agrees with deck
HR,2006.0,0,11,0,agrees with deck
LT,2016.0,12,8,0,agrees with deck
LV,2022.0,18,2,0,agrees with deck
MT,2022.0,17,2,1,DIFFERS from deck
RO,2023.0,18,1,0,DIFFERS from deck


Country-years by DRS status:


,count,nunique
drs_status,,
glass_in_drs,130,10
no_drs,512,26
other_materials_only,78,5


glass DRS country-years: 130 | with a deposit value: 36 | of which carried forward from an earlier snapshot: 21


,min,max
geo,,
DE,2018,2023
DK,2021,2023
EE,2019,2023
FI,2021,2023
HR,2019,2023
IS,2019,2023
LT,2019,2023
LV,2022,2023
MT,2022,2023


drs_group
1_drs_before_window            1
2_drs_introduced_in_window     9
3_no_drs_in_window            20
Name: n_countries, dtype: int64

## 7. Save

In [9]:
column_order = [
    "geo", "country", "year", "material",
    # outcome
    RATE, RAW, f"{RATE}_capped", "dq_flag", "rate_excluded",
    "rate_flag", "series_break_flag", "definition_differs_flag", "post_2020_method_period",
    # DRS timing
    "has_drs", "drs_full_year", "years_since_drs", "drs_group", "drs_status", "has_any_drs",
    START, "drs_start_used", "drs_start_deck", "drs_start_check", "drs_note",
    # DRS intensity
    "deposit_max_eur", "deposit_min_eur", "deposit_structure", "deposit_source_year", "deposit_is_snapshot_year",
]
final = panel[column_order]

dictionary = pd.DataFrame(
    [
        ("geo", "Eurostat geo code (2 letters)", "Eurostat"),
        ("country", "Country name", "Eurostat"),
        ("year", "Calendar year, 2000-2023", "Eurostat"),
        ("material", "Packaging material of this file", "constant"),
        (RATE, f"{MATERIAL.capitalize()} packaging recycling rate, %, CLEANED (impossible values set to missing). OUTCOME", "Eurostat CEI_WM020"),
        (RAW, "Rate as published by Eurostat, including values above 100% and constant runs", "Eurostat CEI_WM020"),
        (f"{RATE}_capped", "Raw rate capped at 100%. Only for sensitivity checks", "derived"),
        ("dq_flag", "Why the raw value was flagged: over_100, flat_run, near_zero, manual (empty = fine)", "derived"),
        ("rate_excluded", "True if the raw value was removed from the analysis column", "derived"),
        ("rate_flag", "Eurostat status flag (b/bd/be = break in series, d = definition differs, e = estimated, p = provisional, i = imputed)", "Eurostat"),
        ("series_break_flag", "True if rate_flag signals a break in the time series", "derived"),
        ("definition_differs_flag", "True if rate_flag says the definition differs (d or bd)", "derived"),
        ("post_2020_method_period", "1 for 2020 onwards. Many countries change in 2020, probably a reporting-method change (check Eurostat metadata)", "derived"),
        ("has_drs", f"1 if year >= {START} (this material is in a DRS), else 0. MAIN EXPLANATORY VARIABLE", "hand-coded"),
        ("drs_full_year", f"1 if year > {START} (first full year of operation)", "hand-coded"),
        ("years_since_drs", f"year - {START}; negative = before; empty = this material never in a DRS by 2023", "derived"),
        ("drs_group", "1 = DRS at or before 2000, 2 = introduced 2001-2023, 3 = none by 2023", "derived"),
        ("drs_status", f"{MATERIAL}_in_drs / other_materials_only (country has a DRS that does not cover {MATERIAL}) / no_drs", "derived"),
        ("has_any_drs", "1 if year >= the start of the country's DRS as a whole", "hand-coded"),
        (START, f"Year {MATERIAL} entered the DRS; empty = {MATERIAL} not covered", "Reloop GDB 2024, operators"),
        ("drs_start_used", "Year the DRS as a whole started, per the Global Deposit Book 2024", "Reloop GDB 2024"),
        ("drs_start_deck", "Start year in the older Reloop deck", "Reloop deck"),
        ("drs_start_check", "Whether the GDB 2024 year agrees with the deck", "derived"),
        ("drs_note", "Free-text note on the DRS (scope changes, conflicts, assumptions)", "hand-coded"),
        ("deposit_max_eur", f"Highest deposit tier on {MATERIAL} in euro; 0 when {MATERIAL} is not in a DRS. Snapshot values carried forward. KNOWN ONLY FROM ~2018. Approximate FX", "Reloop 2020/2022 editions"),
        ("deposit_min_eur", f"Lowest deposit tier on {MATERIAL} in euro. Empty for variable systems before the 2022 edition", "Reloop 2020/2022 editions"),
        ("deposit_structure", "flat, variable or variable (range)", "Reloop 2020/2022 editions"),
        ("deposit_source_year", "Year of the snapshot the deposit value comes from", "derived"),
        ("deposit_is_snapshot_year", "True if the deposit value is for this exact year; False = carried forward", "derived"),
    ],
    columns=["column", "description", "source"],
)
assert set(final.columns) == set(dictionary["column"]), set(final.columns) ^ set(dictionary["column"])

final.to_csv(OUT_DIR / f"drs_recycling_panel_{MATERIAL}.csv", index=False, encoding="utf-8")
snap.to_csv(OUT_DIR / f"drs_deposit_snapshots_{MATERIAL}.csv", index=False, encoding="utf-8")
dictionary.to_csv(OUT_DIR / f"drs_recycling_panel_{MATERIAL}_dictionary.csv", index=False, encoding="utf-8")
print("Saved to", OUT_DIR.resolve(), "|", final.shape)

Saved to /Users/leannericard/mgt499_project/data | (720, 29)


## 8. First look (starting point for the univariate / bivariate work)

In [10]:
final[RATE].describe()

count    590.000000
mean      64.957797
std       22.630955
min        3.500000
25%       49.950000
50%       69.500000
75%       83.225000
max       99.900000
Name: glass_recycling_rate, dtype: float64

In [11]:
# Average (cleaned) recycling rate by DRS status over time
final.pivot_table(index="year", columns="drs_status", values=RATE, aggfunc="mean").round(1)

drs_status,glass_in_drs,no_drs,other_materials_only
year,,,
2000,NaN,60.4,75.2
2001,NaN,60.6,66.8
2002,89.9,61.4,68.5
2003,90.6,58.5,76.5
2004,81.5,46.2,54.9
2005,77.3,46.2,79.2
2006,65.4,48.8,82.7
2007,72.9,48.3,90.2
2008,64.0,54.6,86.8


In [12]:
# Does cleaning change the picture? Mean rate by status, raw vs cleaned vs capped
(final.groupby("drs_status")[[RAW, RATE, f"{RATE}_capped"]].mean().round(1))

,glass_recycling_rate_raw,glass_recycling_rate,glass_recycling_rate_capped
drs_status,,,
glass_in_drs,69.5,74.6,67.6
no_drs,61.5,59.9,61.4
other_materials_only,84.3,83.4,84.1


In [13]:
# Deposit vs recycling rate, DRS country-years with a known deposit (roughly 2018-2023)
dd = final[final["has_drs"] == 1].dropna(subset=["deposit_max_eur", RATE])
print("n =", len(dd), "| countries =", dd["geo"].nunique())
dd[["deposit_max_eur", RATE]].corr().round(2)

n = 30 | countries = 9


,deposit_max_eur,glass_recycling_rate
deposit_max_eur,1.00,0.31
glass_recycling_rate,0.31,1.00


## What this dataset can and cannot answer

**Can**
- **DRS vs no DRS for glass**, 30 countries over 24 years, and before/after comparisons for the countries that introduced a DRS covering glass inside the window (see the pre/post table in the checks).
- **Within-country comparison**: countries with a DRS that does not cover glass (`drs_status = other_materials_only`) are a different comparison group from countries with no DRS at all.

**Cannot (yet)**
- **Other financial incentives** (landfill taxes, packaging taxes, EPR fees) are not in the data.
- **Deposit changes over time**: deposits are unchanged between snapshots almost everywhere, and for variable systems the glass deposit is only known from 2021.
- **Drink containers vs all glass packaging**: the outcome includes non-drink packaging.
- **Causal claims**: few treated countries and short post-periods for 2022–23 launches, so treat before/after results as exploratory.

**Flags to remember:** the 2020 reporting-method break, approximate FX rates, the disputed start years (NL, MT, RO, PT), and the exclusions made in the cleaning step: check that a country you care about has not lost most of its years.